In [14]:

from datetime import datetime, timezone
import catalogo as cat
import funciones_base as fb
import analisis as an
import satelite as sat
from analisis import resumen_por_grupo
from analisis import en_zona, ordenar_por

## 1. Carga de datos

Se lee el archivo `datos/satelites.json` y se calcula la fecha media de los datos con `epoch_medio`. Luego se crean los objetos `Satelite` para un momento de cálculo y se cuenta cuántos registros descarta `crear_satelites` (los que están por debajo de 100 km de altitud).

In [2]:
registros = cat.leer_satelites("datos/satelites.json")
epoch = fb.epoch_medio(registros)
momento = datetime.now(timezone.utc)
satelites = sat.crear_satelites(registros, momento)

print(f"Satélites en el archivo: {len(registros)}")
print(f"Fecha media de los datos (epoch_medio): {epoch}")
print(f"Momento del cálculo: {momento}")
print(f"Días transcurridos: {(momento - epoch).days}")
print(f"Satélites descartados por crear_satelites: {len(registros) - len(satelites)}")


Satélites en el archivo: 16710
Fecha media de los datos (epoch_medio): 2026-10-14 06:30:18.482258+00:00
Momento del cálculo: 2026-10-28 22:33:49.255489+00:00
Días transcurridos: 14
Satélites descartados por crear_satelites: 23


## 2. Grupos

Se cuenta cuántos satélites pertenecen a cada grupo (Starlink, OneWeb, GPS, etc.) con `contar_por_grupo`, y se calcula qué porcentaje del total es Starlink.

In [3]:
conteo_grupos = cat.contar_por_grupo(registros)

for nombre, cantidad in sorted(conteo_grupos.items(), key=lambda par: par[1], reverse=True):
    print(f"{nombre}: {cantidad}")

porcentaje_starlink = conteo_grupos["Starlink"] / len(registros) * 100
print(f"\nStarlink representa el {porcentaje_starlink:.1f} % del total")

Starlink: 11125
Otros: 4075
OneWeb: 651
Kuiper: 391
Qianfan: 257
Iridium: 80
BeiDou: 55
GPS: 40
Globalstar: 36

Starlink representa el 66.6 % del total


## 3. Lanzamientos por año

Se cuenta cuántos de los satélites activos hoy fueron lanzados cada año desde 2015, con `lanzamientos_por_anio`, y se identifica el año con más lanzamientos.

In [4]:
por_anio = cat.lanzamientos_por_anio(registros)
desde_2015 = {anio: n for anio, n in sorted(por_anio.items()) if anio >= 2015}

for anio, cantidad in desde_2015.items():
    print(f"{anio}: {cantidad}")

anio_max = max(desde_2015, key=desde_2015.get)
print(f"\nAño con más lanzamientos: {anio_max} ({desde_2015[anio_max]} satélites)")

2015: 96
2016: 83
2017: 136
2018: 194
2019: 145
2020: 360
2021: 994
2022: 1718
2023: 2266
2024: 2429
2025: 4289
2026: 3285

Año con más lanzamientos: 2025 (4289 satélites)


# 4. Tipos de órbita

Se realiza el conteo de satélites según su tipo de órbita (LEO, MEO, GEO, HEO) utilizando la función `contar_por_tipo` del módulo `analisis`.

In [6]:
conteo_orbitas = an.contar_por_tipo(satelites)

for tipo, cantidad in conteo_orbitas.items():
    print(f"{tipo}: {cantidad}")

LEO: 15879
MEO: 200
GEO: 586
HEO: 22


# 5. Extremos

Se identifican el satélite más bajo, el más alto y el top 10 de cada extremo mediante las funciones `mas_bajo`, `mas_alto` y `top_n` del módulo `analisis`, imprimiendo la información de cada objeto a través de su método `__str__`.

In [7]:
sat_mas_bajo = an.mas_bajo(satelites)
sat_mas_alto = an.mas_alto(satelites)

print("Satélite más bajo:")
print(sat_mas_bajo)

print("\nSatélite más alto:")
print(sat_mas_alto)

top_10_bajos = an.top_n(satelites, 10, mas_altos=False)
top_10_altos = an.top_n(satelites, 10, mas_altos=True)

print("\nTop 10 más bajos:")
for sat in top_10_bajos:
    print(sat)

print("\nTop 10 más altos:")
for sat in top_10_altos:
    print(sat)

Satélite más bajo:
STARLINK-4454 | LEO | 103.61 km

Satélite más alto:
TESS | HEO | 343156.303 km

Top 10 más bajos:
STARLINK-4454 | LEO | 103.61 km
STARLINK-36255 | LEO | 107.59 km
STARLINK-30924 | LEO | 114.185 km
IPERDRONE.0 | LEO | 120.989 km
STARLINK-30224 | LEO | 127.219 km
COSMOS 2572 | LEO | 129.611 km
STARLINK-5823 | LEO | 133.154 km
STARLINK-40075 | LEO | 140.067 km
STARLINK-30236 | LEO | 152.911 km
STARLINK-1454 | LEO | 159.691 km

Top 10 más altos:
TESS | HEO | 343156.303 km
MMS 4 | HEO | 139685.895 km
MMS 3 | HEO | 139661.106 km
MMS 2 | HEO | 139651.018 km
MMS 1 | HEO | 139631.776 km
XMM-NEWTON | HEO | 91503.793 km
THEMIS A | HEO | 77152.307 km
SMILE | HEO | 75588.353 km
CXO | HEO | 75234.928 km
ASTRANIS UTILITYSAT | HEO | 59431.554 km


## 6. Comparación de grupos

Resumen de altitudes (en km) de cada grupo: cantidad de satélites, altitud mínima, media y máxima.

In [13]:

resumen = resumen_por_grupo(satelites)

print("Grupo | Cantidad | Mínima (km) | Media (km) | Máxima (km)")
print("-----------------------------------------------------")

for nombre in resumen:
    datos = resumen[nombre]
    print(nombre, "|", datos["cantidad"], "|", datos["minima"], "|", datos["media"], "|", datos["maxima"])

Grupo | Cantidad | Mínima (km) | Media (km) | Máxima (km)
-----------------------------------------------------
Otros | 4070 | 108.838 | 6451.926904422604 | 340529.716
GPS | 40 | 19716.856 | 20256.33625 | 21044.451
Globalstar | 36 | 687.222 | 1295.4593611111113 | 1835.66
BeiDou | 55 | 21489.651 | 27233.92841818182 | 36085.962
Iridium | 80 | 635.335 | 777.068325 | 807.438
OneWeb | 651 | 548.919 | 1210.1227096774194 | 1251.685
Starlink | 11106 | 109.569 | 468.22411111111114 | 596.33
Qianfan | 257 | 786.874 | 1044.2308365758756 | 1224.707
Kuiper | 391 | 448.907 | 632.0463017902813 | 658.499


## 7. Satélites sobre Bariloche

Se cuentan los satélites que están dentro de la zona BARILOCHE en el momento del cálculo y se los lista de menor a mayor altitud.

La zona es un rectángulo definido por cuatro límites (longitud mínima, latitud mínima, longitud máxima y latitud máxima). Los límites se obtuvieron dibujando el rectángulo sobre el mapa en http://bboxfinder.com, cuyo campo "Box" muestra los cuatro valores en ese mismo orden.

In [15]:
BARILOCHE = (-76.30, -46.13, -66.30, -36.13)   # lon_min, lat_min, lon_max, lat_max

sobre_bariloche = en_zona(satelites, BARILOCHE)
ordenados = ordenar_por(sobre_bariloche, "altitud_km")

print("Satélites sobre Bariloche:", len(ordenados))
print("Ordenados de menor a mayor altitud:")
print("-----------------------------------")

for sat in ordenados:
    print(sat)

Satélites sobre Bariloche: 45
Ordenados de menor a mayor altitud:
-----------------------------------
STARLINK-11550 [DTC] | LEO | 341.948 km
STARLINK-1246 | LEO | 438.278 km
STARLINK-3553 | LEO | 472.395 km
STARLINK-35900 | LEO | 472.514 km
STARLINK-33970 | LEO | 473.896 km
STARLINK-37275 | LEO | 474.57 km
STARLINK-3957 | LEO | 474.67 km
STARLINK-35625 | LEO | 475.999 km
STARLINK-4214 | LEO | 476.073 km
STARLINK-37270 | LEO | 476.25 km
STARLINK-32001 | LEO | 476.882 km
STARLINK-35612 | LEO | 476.937 km
STARLINK-3653 | LEO | 478.104 km
STARLINK-37208 | LEO | 478.194 km
STARLINK-4557 | LEO | 478.448 km
STARLINK-4274 | LEO | 478.751 km
STARLINK-37418 | LEO | 480.166 km
STARLINK-37302 | LEO | 480.872 km
STARLINK-32047 | LEO | 481.785 km
STARLINK-34848 | LEO | 482.181 km
STARLINK-34866 | LEO | 483.054 km
STARLINK-4509 | LEO | 483.771 km
STARLINK-4539 | LEO | 484.803 km
STARLINK-35579 | LEO | 491.236 km
STARLINK-35914 | LEO | 491.541 km
STARLINK-30282 | LEO | 493.544 km
STARLINK-36473 | LEO